# A3 multi-seed study (B1 / R2 / R3 / R4) — PR-CLIP-Improved

Notebook này tạo **toàn bộ số liệu mà thầy yêu cầu bổ sung** cho Bảng A3 của report:

1. **Epoch 0 (không train thêm)**: checkpoint RET-2 (GeoRSCLIP RS5M ViT-B/32) có / không có RIP-fixed, trên RSICD test, RSITMD test và tập validation riêng. Kèm thời gian truy vấn IT(s).
2. **Train 1 epoch × 3 seed** cho B1, R2, R3 (R4 tùy chọn), chọn epoch 0 hay 1 bằng **validation**, test chỉ dùng để báo cáo cuối.
3. Tổng hợp **mean ± std** → `results/a3/summary.csv` và `report/tables/a3_multiseed_rows.tex`.

Code: repo tác giả `ADMIS-TONGJI/PR-CLIP` @ `242fba6` + `prclip_patches/apply_patches.py`. Module chỉ bật/tắt bằng biến môi trường `PRCLIP_RIP`, `PRCLIP_E1`, `PRCLIP_EXT2`.

**Chạy được nhiều phiên Colab**: kết quả lưu trên Drive (`LOGS`), run nào đã xong sẽ được bỏ qua. Cần GPU runtime. Ước tính: ~25–40 phút/run train trên T4/L4, tổng 9 run ≈ 5–6 giờ.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
LOGS = '/content/drive/MyDrive/a3_logs'          # nơi lưu log + metrics.jsonl
IMPROVED = '/content/PR-CLIP-Improved'
REPO = '/content/PR-CLIP'
SEEDS = [1, 2, 3]
TRAIN_CFGS = ['B1', 'R2', 'R3']                   # thêm 'R4' nếu còn thời gian
import os; os.makedirs(LOGS, exist_ok=True)
!nvidia-smi | head -12

In [ ]:
%cd /content
!rm -rf {REPO} {IMPROVED}
!git clone -q https://github.com/hoangk64-ai/PR-CLIP-Improved {IMPROVED}
!git clone -q https://github.com/ADMIS-TONGJI/PR-CLIP {REPO}
!cd {REPO} && git checkout -q 242fba67 && git log --oneline -1
!python {IMPROVED}/prclip_patches/apply_patches.py {REPO}

In [ ]:
!pip install -q open_clip_torch==2.20.0 clip-benchmark==1.6.1 transformers==4.45.2 adapters timm sentence-transformers wandb pyyaml gdown pytest
!pip install -q "protobuf>=6.31.1,<7"
!pip install -q cn_clip --no-deps
!pip install -q faiss-cpu click

In [ ]:
%cd {REPO}
# Ảnh RSICD + RSITMD (cùng link với các notebook cũ của nhóm)
!gdown --fuzzy "https://drive.google.com/file/d/15htMe2iE4kVujjWURxfCxZXk8vAAetoT/view" -O images.zip
!rm -rf images && mkdir images && tar -xzf images.zip -C images
!find images -type f -exec mv -t images {} + 2>/dev/null; find images -mindepth 1 -type d -empty -delete; rm images.zip
# Checkpoint RET-2 = GeoRSCLIP RS5M ViT-B/32 fine-tuned on RET-2
!mkdir -p cache/weights && wget -q -O cache/weights/RET-2.pt https://huggingface.co/Zilun/GeoRSCLIP/resolve/main/ckpt/RS5M_ViT-B-32_RET-2.pt
!ls -la cache/weights/RET-2.pt && sha256sum cache/weights/RET-2.pt

In [ ]:
%cd {REPO}
# Validation riêng theo ảnh (rsicd_val.csv trùng byte với rsicd_test.csv nên không dùng được)
!python {IMPROVED}/scripts/make_val_split.py --repo {REPO} --val-frac 0.1 --seed 0
!cp dataset/split_manifest.json {LOGS}/split_manifest.json
!cd {IMPROVED} && python -m pytest -q tests

## Phase 1 — Epoch 0 (không train): B1 vs RIP-fixed
Kết quả tất định (không phụ thuộc seed). E1/EXT v2 chỉ tác động lúc train nên epoch 0 của R4/R3 bằng R2.

In [ ]:
import glob, subprocess, os
def done(pattern):
    return any(os.path.getsize(p) > 0 for p in glob.glob(f'{LOGS}/{pattern}*/metrics.jsonl'))

def run(*args):
    cmd = ['python', f'{IMPROVED}/scripts/run_a3.py', *args, '--repo', REPO, '--logs', LOGS]
    print('>>', ' '.join(cmd), flush=True)
    subprocess.run(cmd, check=True)

for cfg in ['B1', 'R2']:
    for data in ['rsicd_test', 'rsitmd_test', 'val']:
        if not done(f'A3_{cfg}_ep0_{data}_s1'):
            run('eval', '--cfg', cfg, '--data', data, '--seed', '1')

## Phase 2 — Train 1 epoch × 3 seed, val trước/sau train, test bằng checkpoint epoch 1
Checkpoint (~1.2 GB) bị xóa sau khi đánh giá test để tiết kiệm Drive.

In [ ]:
import shutil
for cfg in TRAIN_CFGS:
    for seed in SEEDS:
        tag = f'A3_{cfg}_train_s{seed}'
        if not done(tag):
            run('train', '--cfg', cfg, '--seed', str(seed))
        ckpts = sorted(glob.glob(f'{LOGS}/{tag}*/checkpoints/epoch_1.pt'))
        for data in ['rsicd_test', 'rsitmd_test']:
            if not done(f'A3_{cfg}_ep1_{data}_s{seed}'):
                assert ckpts, f'Không thấy checkpoint của {tag}'
                run('eval', '--cfg', cfg, '--data', data, '--seed', str(seed), '--resume', ckpts[-1])
        for c in glob.glob(f'{LOGS}/{tag}*/checkpoints/*.pt'):
            os.remove(c)   # đã đánh giá xong, giải phóng Drive
        print('DONE', cfg, seed, flush=True)

## Phase 3 — Tổng hợp mean ± std

In [ ]:
%cd {IMPROVED}
!python scripts/aggregate_results.py --logs {LOGS} --out {IMPROVED} --test rsicd_test
!python scripts/aggregate_results.py --logs {LOGS} --out /content/rsitmd_summary --test rsitmd_test
!mkdir -p {LOGS}/summary && cp -r results/a3 report/tables {LOGS}/summary/ && cp -r /content/rsitmd_summary/results/a3 {LOGS}/summary/a3_rsitmd
# Gom log text (out.log + metrics.jsonl) để đưa lên repo
!mkdir -p results/a3/raw_logs && for d in {LOGS}/A3_*; do n=$(basename $d); mkdir -p results/a3/raw_logs/$n; cp $d/out.log $d/metrics.jsonl $d/params.yml results/a3/raw_logs/$n/ 2>/dev/null; done
!cp -r results/a3/raw_logs {LOGS}/summary/
!cat results/a3/summary.md

## Đưa kết quả lên repo
Tải thư mục `a3_logs/summary` trên Drive về, chép vào repo `PR-CLIP-Improved` (giữ nguyên đường dẫn `results/a3/...`, `report/tables/...`), rồi commit + push. Sau đó build lại report: `cd report && latexmk -pdf main.tex`.